# 🏷️ 01 - Types of Data: What a Column *Means* vs. How It's Stored

In Module 06 you checked `.dtypes` and fixed columns that had the wrong one. That answered one question: **how does pandas store this column?**

This module starts with a second, different question: **what does this column actually measure?** The answer decides which summaries make sense, and (in Module 08) which chart to draw.

```text
dtype                →  how pandas stores the values   (int64, float64, str, bool, category)
measurement scale    →  what the values mean           (nominal, ordinal, interval, ratio)
```

A column can be stored as a number and still be a category. Titanic's `Pclass` is stored as `1`, `2`, `3`, but "the average passenger class is 2.31" is not a sentence anyone should say.

## ✅ Learning Goals
- Tell the difference between a column's **dtype** and its **measurement scale**
- Classify a variable as qualitative or quantitative, and discrete or continuous
- Place a variable on the Nominal / Ordinal / Interval / Ratio scale, and say what math is allowed on it
- Tell pandas what a column means with `category`, ordered categories, and `bool`
- Build and save a **data dictionary** for a dataset

## 0. Setup
This notebook saves one file (a data dictionary) into **`01-OutputFiles`**, the same pattern as Foundations and the Module 06 lab.

In [1]:
import pandas as pd
from pathlib import Path

OUT = Path('01-OutputFiles')
OUT.mkdir(exist_ok=True)

titanic = pd.read_csv('titanic.csv')
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1. Storage Is Not Meaning
Start the way Module 06 taught you: look before you do anything.

In [2]:
titanic.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

> 📝 **pandas 3 note:** text columns now show the dtype `str`. Older tutorials (and older pandas) show `object` for the same thing. Treat them the same way.

Now read the numeric columns with the *meaning* in mind:

| Column | dtype | What it really is |
|---|---|---|
| `PassengerId` | `int64` | An **ID**. A label that happens to be a number. Its mean is meaningless. |
| `Survived` | `int64` | A **yes/no** outcome stored as 0/1. |
| `Pclass` | `int64` | A **ranked category**: 1st, 2nd, 3rd class. |
| `Age` | `float64` | A real **measurement**. |
| `SibSp` | `int64` | A real **count** (siblings/spouses aboard). |

Pandas will happily compute statistics on all of them, including the ones that make no sense:

In [3]:
titanic[['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp']].mean()

PassengerId    446.000000
Survived         0.383838
Pclass           2.308642
Age             29.699118
SibSp            0.523008
dtype: float64

Every number above was computed correctly. Only some of them *mean* anything:
- Mean `PassengerId` (446): nonsense. IDs are labels.
- Mean `Pclass` (2.31): misleading. The gap between 1st and 2nd class isn't "one unit" of anything.
- Mean `Survived` (0.38): actually useful! The mean of a 0/1 column is the **proportion** of 1s, so 38% survived. (More on this in section 5.)
- Mean `Age` and `SibSp`: fine.

> ⚡ **Code won't stop you from computing nonsense. Only understanding the data will.**

## 2. Qualitative vs. Quantitative, Discrete vs. Continuous

**Qualitative** (categorical) variables put each row in a **group**: `Sex`, `Embarked`, `Pclass`.
**Quantitative** (numeric) variables **measure or count** something: `Age`, `Fare`, `SibSp`.

Quantitative variables split again:

| Type | Definition | Titanic examples | Usual dtype |
|---|---|---|---|
| **Discrete** | Countable, with gaps between possible values | `SibSp`, `Parch` | `int64` / `Int64` |
| **Continuous** | Can fall anywhere in a range | `Age`, `Fare` | `float64` |

A quick test for discrete: *can you have 2.5 of it?* You can't have 2.5 siblings, but you can be 2.5 years old.

In [4]:
# Discrete: a small set of whole-number values
print(titanic['SibSp'].value_counts().sort_index())

# Continuous: so many distinct values that counting each one isn't useful
print('Distinct ages:', titanic['Age'].nunique())

SibSp
0    608
1    209
2     28
3     16
4     18
5      5
8      7
Name: count, dtype: int64
Distinct ages: 88


### 🧠 Quick Check
`Age` is continuous, but this file stores most ages as whole numbers (`22.0`, `38.0`). Does rounding change what the variable *measures*? (Hint: rounding is a recording choice, not a property of age.)

## 3. The Four Scales of Measurement

| Scale | Kind | Ordered? | Equal gaps? | True zero? | Titanic example | Allowed math |
|---|---|:-:|:-:|:-:|---|---|
| **Nominal** | Categorical | ❌ | ❌ | ❌ | `Sex`, `Embarked` | counts, %, mode |
| **Ordinal** | Categorical | ✅ | ❌ | ❌ | `Pclass` | + median, percentiles, ranking |
| **Interval** | Numeric | ✅ | ✅ | ❌ | *(none in Titanic; °C, calendar year)* | + differences, mean, std |
| **Ratio** | Numeric | ✅ | ✅ | ✅ | `Age`, `Fare`, `SibSp` | + ratios ("twice as much") |

Each scale allows everything the scale above it allows, plus a bit more.

- **Interval vs. Ratio** hinges on whether zero means "none of it." `$0` fare means no money paid, so `$100` really is twice `$50`. `0°C` is just where water freezes, so 20°C is not "twice as hot" as 10°C.
- **Ordinal** is the sneaky one. It *looks* numeric (`1`, `2`, `3`; or 1–5 stars), but the gaps aren't guaranteed equal. Is the step from 1 star to 2 stars the same as 4 to 5?

✅ **Your Turn:** Fill in the scale for each variable, then re-run the cell.

In [6]:
variables = {
    'Embarked (C, Q, S)': 'Nominal',
    'Ticket number': 'Ordinal',
    'Customer rating, 1-5 stars': 'Ordinal',
    'Temperature in Fahrenheit': 'Interval',
    'Fare in dollars': 'Ratio',
    'Year a building was built': 'Interval',
}
for name, scale in variables.items():
    print(f'{name:<30} -> {scale}')

Embarked (C, Q, S)             -> Nominal
Ticket number                  -> Ordinal
Customer rating, 1-5 stars     -> Ordinal
Temperature in Fahrenheit      -> Interval
Fare in dollars                -> Ratio
Year a building was built      -> Interval


## 4. Telling Pandas What a Column Means
Pandas can't guess meaning, but you can **tell** it. Module 06 used the `category` dtype to save memory. Here it does a more important job: it records that a column is a set of groups, which plotting and summary tools will respect later.

### Nominal → `category`

In [7]:
titanic['Embarked'] = titanic['Embarked'].astype('category')
print(titanic['Embarked'].dtype)
print(titanic['Embarked'].cat.categories)

category
Index(['C', 'Q', 'S'], dtype='str')


### Ordinal → an *ordered* category
For an ordinal column, also tell pandas the **order**. First turn the codes into readable labels (`.map()` from Module 06), then define the order from lowest to highest:

In [8]:
class_labels = {1: '1st', 2: '2nd', 3: '3rd'}
class_order = pd.CategoricalDtype(['3rd', '2nd', '1st'], ordered=True)   # lowest → highest

titanic['class'] = titanic['Pclass'].map(class_labels).astype(class_order)
titanic['class'].head()

0    3rd
1    1st
2    3rd
3    1st
4    3rd
Name: class, dtype: category
Categories (3, str): ['3rd' < '2nd' < '1st']

The `<` signs in the output show that pandas now knows the order. That unlocks things a plain text column can't do:

In [9]:
print('Lowest class: ', titanic['class'].min())
print('Highest class:', titanic['class'].max())

# Comparisons follow YOUR order, not alphabetical order
upper_classes = titanic[titanic['class'] > '3rd']
print('Passengers above 3rd class:', len(upper_classes))

# Counts come back in the logical order, not by frequency or alphabet
titanic['class'].value_counts(sort=False)

Lowest class:  3rd
Highest class: 1st
Passengers above 3rd class: 400


class
3rd    491
2nd    184
1st    216
Name: count, dtype: int64

⚠️ Without the ordered dtype, `'1st' < '3rd'` is just an **alphabetical** comparison, which happens to work here by luck. Try `'10th' < '2nd'` in a cell: alphabetically, it's `True`.

### Yes/No → `bool`

In [10]:
titanic['survived'] = titanic['Survived'].astype(bool)
titanic['survived'].value_counts()

survived
False    549
True     342
Name: count, dtype: int64

## 5. Why the Mean of a Yes/No Column Is a Proportion
`True` counts as `1` and `False` as `0`. So the mean is (number of `True`) ÷ (number of rows), which is the **proportion** of `True`. Multiply by 100 for a percentage.

In [11]:
survival_rate = titanic['survived'].mean()
print(f'Survival rate: {survival_rate:.1%}')

# The same trick works on any True/False mask from Module 06
print(f'Share of passengers under 18: {(titanic["Age"] < 18).mean():.1%}')

Survival rate: 38.4%
Share of passengers under 18: 12.7%


⚠️ Look closely at that second number. `Age` has missing values, and `NaN < 18` is `False`, so missing ages got counted as "not under 18." The share among passengers **with a known age** is higher:

In [12]:
known_age = titanic['Age'].dropna()
print(f'Share under 18 (known ages only): {(known_age < 18).mean():.1%}')

Share under 18 (known ages only): 15.8%


Which one is "right" depends on the question, but you should always know **which one you computed**. This is Module 06's habit (inspect missing values first) paying off.

## 6. Choosing Summaries by Type
This table is the map for the rest of the module (and the chart choices in Module 08):

| The column is… | Center | Spread | Useful summaries | Coming up |
|---|---|---|---|---|
| Nominal | mode | (number of categories) | `value_counts(normalize=True)`, `nunique()` | notebook 04 |
| Ordinal | median, mode | percentiles | counts **in order**, percentiles | notebooks 02, 04 |
| Interval / Ratio | mean, median | std, IQR, range | `describe()`, quantiles, skew | notebooks 02, 03 |
| Boolean | proportion (mean) | — | `.mean()`, rates by group | notebook 04 |
| Two numeric columns | — | — | correlation | notebook 05 |

## 7. Build a Data Dictionary
A **data dictionary** is a table describing every column: its dtype, what it means, and its scale. Real teams keep one for every dataset. Writing one forces you to answer "what does this column mean?" for every column, before you analyze anything.

In [13]:
data_dictionary = pd.DataFrame({
    'column': ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Fare', 'Embarked'],
    'meaning': ['Row ID', '1 = survived', 'Ticket class', 'Sex', 'Age in years',
                'Siblings/spouses aboard', 'Ticket price (pounds)', 'Port of embarkation'],
    'kind': ['identifier', 'qualitative', 'qualitative', 'qualitative', 'quantitative',
             'quantitative', 'quantitative', 'qualitative'],
    'scale': ['nominal', 'nominal', 'ordinal', 'nominal', 'ratio', 'ratio', 'ratio', 'nominal'],
})

# Add the facts pandas can measure for us
raw = pd.read_csv('titanic.csv')
data_dictionary['dtype'] = data_dictionary['column'].map(raw.dtypes.astype(str))
data_dictionary['missing'] = data_dictionary['column'].map(raw.isna().sum())
data_dictionary['distinct'] = data_dictionary['column'].map(raw.nunique())

data_dictionary.to_csv(OUT / 'titanic_data_dictionary.csv', index=False)
data_dictionary

,column,meaning,kind,scale,dtype,missing,distinct
0,PassengerId,Row ID,identifier,nominal,int64,0,891
1,Survived,1 = survived,qualitative,nominal,int64,0,2
2,Pclass,Ticket class,qualitative,ordinal,int64,0,3
3,Sex,Sex,qualitative,nominal,str,0,2
4,Age,Age in years,quantitative,ratio,float64,177,88
5,SibSp,Siblings/spouses aboard,quantitative,ratio,int64,0,7
6,Fare,Ticket price (pounds),quantitative,ratio,float64,0,248
7,Embarked,Port of embarkation,qualitative,nominal,str,2,3


Notice the `distinct` column. `Survived` has 2 distinct values and `Pclass` has 3: a numeric column with only a handful of distinct values is often a category in disguise. That's a good first clue, but not proof. You still have to think about what the column measures.

---
### 📝 Summary
- **dtype** is how pandas stores a column. **Measurement scale** is what it means. They're separate questions.
- Nominal → ordinal → interval → ratio: each level allows more math than the one before.
- Numbers can be categories (IDs, class codes, 0/1 outcomes). Ask before you average.
- Use `category`, ordered `CategoricalDtype`, and `bool` to record a column's meaning in pandas.
- The mean of a True/False column is a proportion, but check how missing values were counted.

## 🏋️ Practice

### Practice 1 — Classify Pokémon Columns (easy)
Load `pokemon.csv` (in this folder) into `poke`. For `#`, `Type 1`, `HP`, `Generation`, and `Legendary`, write the dtype pandas chose, and the measurement scale you think is correct (as a comment or a small dictionary).

In [ ]:
poke = pd.read_csv('pokemon.csv')
# Your code here
poke[['#', 'Type 1', 'HP', 'Generation', 'Legendary']].dtypes
# # -> int64, Ordinal
# Type 1 -> str, Nominal
# HP -> int64, Ratio
# Generation -> int64, Ordinal
# Legendary -> bool, Nominal

#             int64
Type 1          str
HP            int64
Generation    int64
Legendary      bool
dtype: object

### Practice 2 — Give Pandas the Meaning (medium)
1. Convert `Generation` to an **ordered** category (1 through 6).
2. Print the share of Pokémon that are `Legendary`, as a percentage.
3. Use your ordered `Generation` to count how many Pokémon come from generation 4 or later (hint: compare with `>=`).

In [15]:
# Your code here
# Convert `Generation` to an **ordered** category (1 through 6).
poke['Generation'] = pd.Categorical(
    poke['Generation'],
    categories=[1, 2, 3, 4, 5, 6],
    ordered=True,
)

# Print the share of Pokémon that are `Legendary`, as a percentage.
print(poke['Legendary'].mean() * 100)

# Use your ordered `Generation` to count how many Pokémon come from generation 4 or later (hint: compare with `>=`).
print((poke['Generation'] >= 4).sum())

8.125
368


### Practice 3 — A Column Describer (harder)
Write a function `describe_column(series)` that returns a dictionary with the column's `dtype`, number of missing values, and number of distinct values. Then add a `'suggested_summary'`:
- `'proportion'` if the dtype is `bool`
- `'counts'` if the column has 10 or fewer distinct values
- `'mean/median/std'` otherwise

Run it on every column of `titanic` (hint: loop over `titanic.columns`). Which suggestions are wrong, and why can't a function catch them?

In [18]:
# Your code here
def describe_column(series):
    distinct_values = series.nunique(dropna=False)
    if series.dtype == 'bool':
        suggested_summary = 'proportion'
    elif distinct_values <= 10:
        suggested_summary = 'counts'
    else:
        suggested_summary = 'mean/median/std'

    return {
        'dtype': series.dtype,
        'missing': series.isna().sum(),
        'distinct': distinct_values,
        'suggested_summary': suggested_summary,
    }


for column in titanic.columns:
    print(column, describe_column(titanic[column]))

PassengerId {'dtype': dtype('int64'), 'missing': np.int64(0), 'distinct': 891, 'suggested_summary': 'mean/median/std'}
Survived {'dtype': dtype('int64'), 'missing': np.int64(0), 'distinct': 2, 'suggested_summary': 'counts'}
Pclass {'dtype': dtype('int64'), 'missing': np.int64(0), 'distinct': 3, 'suggested_summary': 'counts'}
Name {'dtype': <StringDtype(storage='python', na_value=nan)>, 'missing': np.int64(0), 'distinct': 891, 'suggested_summary': 'mean/median/std'}
Sex {'dtype': <StringDtype(storage='python', na_value=nan)>, 'missing': np.int64(0), 'distinct': 2, 'suggested_summary': 'counts'}
Age {'dtype': dtype('float64'), 'missing': np.int64(177), 'distinct': 89, 'suggested_summary': 'mean/median/std'}
SibSp {'dtype': dtype('int64'), 'missing': np.int64(0), 'distinct': 7, 'suggested_summary': 'counts'}
Parch {'dtype': dtype('int64'), 'missing': np.int64(0), 'distinct': 7, 'suggested_summary': 'counts'}
Ticket {'dtype': <StringDtype(storage='python', na_value=nan)>, 'missing': np.int

*Which suggestions were wrong, and why?*

`PassengerId` and `Ticket` have many distinct values, so the function suggests `mean/median/std`, but they are identifiers rather than numeric measurements.

`Cabin` has 10 or fewer distinct values, so it suggests `counts`, but its values are mostly unique and not a useful categorical summary.

A function can inspect dtype and distinct-value counts, but it cannot know whether a column is an identifier, a measurement, or a sparse categorical variable.

## ✏️ Your Turn
Build a data dictionary for `pokemon.csv` with columns `column`, `meaning`, `kind`, `scale`, `dtype`, `missing`, and `distinct`, covering **every** column. Save it as `OUT / 'pokemon_data_dictionary.csv'`.

In [20]:
# Your turn here
# Build a data dictionary for `pokemon.csv` with columns `column`, `meaning`, `kind`, `scale`, `dtype`, `missing`, and `distinct`, covering **every** column. Save it as `OUT / 'pokemon_data_dictionary.csv'`.
pokemon_data_dictionary = pd.DataFrame({
    'column': ['Name', 'Type 1', 'Type 2', 'Total', 'HP', 'Attack', 'Defense', 'Sp. Atk',
               'Sp. Def', 'Speed', 'Generation', 'Legendary'],
    'meaning': ['Pokémon name', 'Primary type', 'Secondary type', 'Total base stats', 'Hit points',
                'Attack stat', 'Defense stat', 'Special attack stat', 'Special defense stat',
                'Speed stat', 'Game generation', 'Whether the Pokémon is legendary'],
    'kind': ['identifier', 'qualitative', 'qualitative', 'quantitative', 'quantitative',
             'quantitative', 'quantitative', 'quantitative', 'quantitative', 'quantitative',
             'qualitative', 'qualitative'],
    'scale': ['nominal', 'nominal', 'nominal', 'ratio', 'ratio', 'ratio', 'ratio', 'ratio',
              'ratio', 'ratio', 'ordinal', 'nominal'],
})

# Add the facts pandas can measure for us
raw = pd.read_csv('pokemon.csv')
pokemon_data_dictionary['dtype'] = pokemon_data_dictionary['column'].map(raw.dtypes.astype(str))
pokemon_data_dictionary['missing'] = pokemon_data_dictionary['column'].map(raw.isna().sum())
pokemon_data_dictionary['distinct'] = pokemon_data_dictionary['column'].map(raw.nunique())

pokemon_data_dictionary.to_csv(OUT / 'pokemon_data_dictionary.csv', index=False)
pokemon_data_dictionary

,column,meaning,kind,scale,dtype,missing,distinct
0,Name,Pokémon name,identifier,nominal,str,0,800
1,Type 1,Primary type,qualitative,nominal,str,0,18
2,Type 2,Secondary type,qualitative,nominal,str,386,18
3,Total,Total base stats,quantitative,ratio,int64,0,200
4,HP,Hit points,quantitative,ratio,int64,0,94
5,Attack,Attack stat,quantitative,ratio,int64,0,111
6,Defense,Defense stat,quantitative,ratio,int64,0,103
7,Sp. Atk,Special attack stat,quantitative,ratio,int64,0,105
8,Sp. Def,Special defense stat,quantitative,ratio,int64,0,92
9,Speed,Speed stat,quantitative,ratio,int64,0,108


## 🔥 Challenge (Optional) — What Does `category` Save?
Module 06 mentioned that `category` saves memory for repeated text. Measure it: compare `titanic['Sex'].memory_usage(deep=True)` with the same column converted to `category`. Then try it on `Name`, where almost every value is different. Why does `category` help one column and not the other?

In [ ]:
# Your code here

## 📦 Check Your Output Files
Run this last. Everything you saved should be inside `01-OutputFiles`. Upload that folder along with this notebook.

In [21]:
for path in sorted(OUT.rglob('*')):
    print(path)

01-OutputFiles\pokemon_data_dictionary.csv
01-OutputFiles\titanic_data_dictionary.csv


---
✅ Next up → [Center and Spread](02-Center_and_Spread.ipynb): the summaries for numeric columns, and why the mean and median can disagree.